# Deep Past Initiative: Data Preprocessing

This notebook covers all data preprocessing steps for the Akkadian → English machine translation pipeline:
1. Setup & imports
2. Load competition and external datasets
3. Normalize transliterations and translations
4. Cross-dataset deduplication
5. Genre prefix conditioning
6. Stage dataset assembly
7. Glossary construction

## 1. Setup & Imports

In [13]:
# pip install -r requirements.txt

In [14]:
import os
import re
import json
import random
import unicodedata

import numpy as np
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Ensure output directories exist
os.makedirs('data', exist_ok=True)
os.makedirs('data/akkademia', exist_ok=True)
os.makedirs('models/stage1_final', exist_ok=True)
os.makedirs('models/stage2_final', exist_ok=True)
os.makedirs('models/stage3_final', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('Setup complete.')

Setup complete.


## 2. Load Datasets

### 2a. Competition Datasets

In [15]:
train_df              = pd.read_csv('data/competition/train.csv')
test_df               = pd.read_csv('data/competition/test.csv')
sample_submission_df  = pd.read_csv('data/competition/sample_submission.csv')
sentences_oare_df     = pd.read_csv('data/competition/Sentences_Oare_FirstWord_LinNum.csv')
published_texts_df    = pd.read_csv('data/competition/published_texts.csv')
lexicon_df            = pd.read_csv('data/competition/OA_Lexicon_eBL.csv')
dictionary_df         = pd.read_csv('data/competition/eBL_Dictionary.csv')

print('Competition datasets loaded.')
print(f'  train:             {train_df.shape}')
print(f'  test:              {test_df.shape}')
print(f'  sample_submission: {sample_submission_df.shape}')
print(f'  sentences_oare:    {sentences_oare_df.shape}')
print(f'  published_texts:   {published_texts_df.shape}')
print(f'  lexicon:           {lexicon_df.shape}')
print(f'  dictionary:        {dictionary_df.shape}')
train_df.head(2)

Competition datasets loaded.
  train:             (1561, 3)
  test:              (4, 5)
  sample_submission: (4, 2)
  sentences_oare:    (9782, 12)
  published_texts:   (7953, 19)
  lexicon:           (39332, 9)
  dictionary:        (19215, 3)


,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...


### 2b. Akkademia Dataset

In [16]:
def load_akkademia_dataset():
    """Load Akkademia NMT_input files and return aligned DataFrames per split."""
    akkademia_data = {}
    for split in ['train', 'valid', 'test']:
        with open(f'data/akkademia/{split}.ak', 'r', encoding='utf-8') as f:
            akkadian_lines = [line.strip() for line in f]
        with open(f'data/akkademia/{split}.en', 'r', encoding='utf-8') as f:
            english_lines = [line.strip() for line in f]
        akkademia_data[split] = pd.DataFrame({
            'transliteration': akkadian_lines,
            'translation': english_lines
        })
    print('Akkademia dataset loaded.')
    for split, df in akkademia_data.items():
        print(f'  {split}: {len(df)} pairs')
    return akkademia_data

akkademia_data = load_akkademia_dataset()
akkademia_data['train'].head(2)

Akkademia dataset loaded.
  train: 50478 pairs
  valid: 2812 pairs
  test: 2870 pairs


,transliteration,translation
0,𒉭𒁄𒌀𒆠𒋗𒄣𒊒𒈾𒉘𒀭...𒀭𒊺𒊒𒀀𒀊𒁀...𒉿𒄘𒀭𒊩𒌆𒃞𒈾𒃻𒀀𒈾𒁁𒂁𒆳𒆳...𒀭𒋾𒀠𒋃......,"Precious scion of Baltil (Aššur), beloved of t..."
1,𒄣𒊏𒁺...𒌑𒊕𒉌𒋙𒊺𒁍𒍑𒋗...𒀸𒄑𒆪𒌑𒌑𒆥𒌅...𒈬𒌓𒊑𒆪...,warrior ... who made ... bow down at his feet ...


### 2c. ORACC Kaggle Dataset

In [17]:
oracc_df = pd.read_csv('data/oracc_kaggle/train.csv')
print(f'ORACC dataset loaded. Shape: {oracc_df.shape}')
print('Columns:', oracc_df.columns.tolist())

# Identify genre priority (used later for sample weighting)
GENRE_PRIORITY = {
    'HIGH':   ['letter', 'administrative', 'memo', 'note', 'decree', 'report'],
    'MEDIUM': ['legal', 'contract', 'list'],
    'LOW':    ['royal inscription', 'votive inscription', 'building inscription', 'annals'],
}
oracc_df.head(2)

ORACC dataset loaded. Shape: (2117, 3)
Columns: ['id', 'akkadian', 'english']


,id,akkadian,english
0,P236874,a - mat LUGAL a - na m d + EN — DÙ DI - mu a ....,The king's word to Bel-ibni : I am well ; you ...
1,P236876,[ x x ] ⸢ x ⸣ É ⸢ x ⸣ + [ x x x x x x x x ] [ ...,[ No ] w then my forces have co [ me and ] ass...


## 3. Text Normalization

In [18]:
# Unicode subscript digits → ASCII
_SUBSCRIPT_MAP = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')

def normalize_transliteration(text):
    """Normalize Akkadian transliteration text.

    Steps:
      - Unicode NFC normalization
      - Subscript digit mapping (₁ → 1, etc.)
      - Replace damage/lacuna markers [...] with <gap>
      - Strip scribal correction markers (!, ?, #)
      - Collapse whitespace
    """
    if pd.isna(text) or text == '':
        return text
    text = unicodedata.normalize('NFC', text)
    text = text.translate(_SUBSCRIPT_MAP)
    text = re.sub(r'\[.*?\]', '<gap>', text)
    text = re.sub(r'[!?#]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_translation(text):
    """Normalize English translation text.

    Steps:
      - Normalize fancy quotation marks to ASCII
      - Normalize whitespace
      - Strip leading/trailing editorial bracket marks
    """
    if pd.isna(text) or text == '':
        return text
    text = re.sub(r'[“”‟\u02ee\uff02]', '"', text)
    text = re.sub(r"[\u2018\u2019\u201b\u02bc\u055a]", "'", text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^[\[\]()]+|[\[\]()]+$', '', text)
    return text


# --- Quick sanity checks ---
assert normalize_transliteration('a₁-šur [broken] text!') == 'a1-šur <gap> text'
assert normalize_translation('\u201cHello\u201d') == '"Hello"'
print('Normalization functions defined and verified.')

Normalization functions defined and verified.


### 3a. Apply Normalization to All Datasets

In [19]:
# Competition train
train_df['transliteration_normalized'] = train_df['transliteration'].apply(normalize_transliteration)
train_df['translation_normalized']     = train_df['translation'].apply(normalize_translation)

# Akkademia splits
for split in ['train', 'valid', 'test']:
    akkademia_data[split]['transliteration_normalized'] = (
        akkademia_data[split]['transliteration'].apply(normalize_transliteration)
    )
    akkademia_data[split]['translation_normalized'] = (
        akkademia_data[split]['translation'].apply(normalize_translation)
    )

# ORACC — columns are 'akkadian' and 'english'
oracc_df['transliteration_normalized'] = oracc_df['akkadian'].apply(normalize_transliteration)
oracc_df['translation_normalized']     = oracc_df['english'].apply(normalize_translation)

print('Normalization applied to all datasets.')
print('Competition train sample:')
train_df[['transliteration', 'transliteration_normalized']].head(2)

Normalization applied to all datasets.
Competition train sample:


,transliteration,transliteration_normalized
0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...
1,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,1 TÚG ša qá-tim i-tur4-DINGIR il5-qé


## 4. Cross-Dataset Deduplication

In [20]:
# Build set of all normalized competition transliterations to exclude from external data
competition_translit_set = set(train_df['transliteration_normalized'].dropna())
print(f'Competition train unique transliterations: {len(competition_translit_set)}')

# --- Akkademia: remove rows that appear in competition train ---
akkademia_deduped = {}
for split in ['train', 'valid', 'test']:
    df = akkademia_data[split]
    before = len(df)
    df = df[~df['transliteration_normalized'].isin(competition_translit_set)].copy()
    df = df.drop_duplicates(subset=['transliteration_normalized'], keep='first')
    after = len(df)
    akkademia_deduped[split] = df
    print(f'  Akkademia {split}: {before} → {after} rows')

# --- ORACC: remove competition overlaps and internal duplicates ---
before = len(oracc_df)
oracc_deduped = oracc_df[
    ~oracc_df['transliteration_normalized'].isin(competition_translit_set)
].drop_duplicates(subset=['transliteration_normalized'], keep='first').copy()
print(f'  ORACC: {before} → {len(oracc_deduped)} rows')

# --- Sentences_Oare: exclude tablets that appear in competition train ---
before = len(sentences_oare_df)
sentences_oare_filtered = (
    sentences_oare_df[~sentences_oare_df['text_uuid'].isin(train_df['oare_id'])]
    .dropna(subset=['translation'])
    .copy()
)
print(f'  Sentences_Oare: {before} → {len(sentences_oare_filtered)} rows (nulls removed)')

Competition train unique transliterations: 1559
  Akkademia train: 50478 → 44511 rows
  Akkademia valid: 2812 → 2712 rows
  Akkademia test: 2870 → 2762 rows
  ORACC: 2117 → 2111 rows
  Sentences_Oare: 9782 → 8565 rows (nulls removed)


## 5. Genre Prefix Conditioning

In [21]:
GENRE_PREFIX_MAP = {
    'letter':         '[LETTER]',
    'debt note':      '[DEBT_NOTE]',
    'loan':           '[DEBT_NOTE]',
    'contract':       '[CONTRACT]',
    'administrative': '[ADMIN]',
    'admin':          '[ADMIN]',
}

def map_genre_to_prefix(genre_label):
    """Map a genre label string to its special prefix token."""
    if pd.isna(genre_label):
        return '[UNKNOWN]'
    g = genre_label.lower().strip()
    for key, prefix in GENRE_PREFIX_MAP.items():
        if key in g:
            return prefix
    return '[UNKNOWN]'


# Merge genre labels from published_texts into competition train
train_with_genre = train_df.merge(
    published_texts_df[['oare_id', 'genre_label']],
    on='oare_id',
    how='left'
)

train_with_genre['genre_prefix'] = train_with_genre['genre_label'].apply(map_genre_to_prefix)
train_with_genre['transliteration_prefixed'] = (
    train_with_genre['genre_prefix'] + ' ' + train_with_genre['transliteration_normalized']
)

print('Genre prefix conditioning applied.')
print('Genre distribution:')
print(train_with_genre['genre_prefix'].value_counts())
train_with_genre[['transliteration_normalized', 'genre_prefix', 'transliteration_prefixed']].head(3)

Genre prefix conditioning applied.
Genre distribution:
genre_prefix
[UNKNOWN]      1392
[LETTER]        150
[ADMIN]          10
[DEBT_NOTE]       5
[CONTRACT]        4
Name: count, dtype: int64


,transliteration_normalized,genre_prefix,transliteration_prefixed
0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,[DEBT_NOTE],[DEBT_NOTE] KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-l...
1,1 TÚG ša qá-tim i-tur4-DINGIR il5-qé,[UNKNOWN],[UNKNOWN] 1 TÚG ša qá-tim i-tur4-DINGIR il5-qé
2,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,[UNKNOWN],[UNKNOWN] TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 ...


## 6. Stage Dataset Assembly

In [22]:
# ── Stage 1: Akkademia (train + valid) + ORACC ──────────────────────────────
akkademia_stage1 = pd.concat([
    akkademia_deduped['train'],
    akkademia_deduped['valid'],
])

stage1_data = pd.concat([
    akkademia_stage1[['transliteration_normalized', 'translation_normalized']],
    oracc_deduped[['transliteration_normalized', 'translation_normalized']],
], ignore_index=True).rename(columns={
    'transliteration_normalized': 'transliteration',
    'translation_normalized':     'translation',
})
stage1_data.dropna(subset=['transliteration', 'translation'], inplace=True)
print(f'Stage 1 dataset: {len(stage1_data)} rows')

# ── Stage 2: Sentences_Oare rows not in competition train ───────────────────
stage2_data = sentences_oare_filtered[
    ['text_uuid', 'translation', 'first_word_spelling', 'line_number', 'side']
].copy()
print(f'Stage 2 dataset: {len(stage2_data)} rows')

# ── Stage 3: Competition train, 90/10 doc-order split ───────────────────────
stage3_full = train_with_genre[[
    'transliteration_prefixed', 'translation_normalized'
]].rename(columns={
    'transliteration_prefixed': 'transliteration',
    'translation_normalized':   'translation',
})

split_idx    = int(len(stage3_full) * 0.9)
stage3_train = stage3_full.iloc[:split_idx].copy()
stage3_val   = stage3_full.iloc[split_idx:].copy()
print(f'Stage 3 train: {len(stage3_train)} rows  |  val: {len(stage3_val)} rows')

# ── Save to disk ─────────────────────────────────────────────────────────────
stage1_data.to_csv('data/stage1_train.csv',  index=False)
stage2_data.to_csv('data/stage2_train.csv',  index=False)
stage3_train.to_csv('data/stage3_train.csv', index=False)
stage3_val.to_csv('data/stage3_val.csv',     index=False)

print('Stage datasets saved to data/.')

Stage 1 dataset: 49334 rows
Stage 2 dataset: 8565 rows
Stage 3 train: 1404 rows  |  val: 157 rows
Stage datasets saved to data/.


## 7. Glossary Construction

In [23]:
# Extract personal names (PN), geographic names (GN), and month names (MN)
# from OA_Lexicon_eBL.csv to build a proper-name glossary.

names_df = lexicon_df[lexicon_df['type'].isin(['PN', 'GN', 'MN'])].copy()
glossary_dict = dict(zip(names_df['form'], names_df['norm']))

print(f'Glossary entries: {len(glossary_dict)}')
print('Sample entries:', dict(list(glossary_dict.items())[:5]))

with open('data/glossary.json', 'w', encoding='utf-8') as f:
    json.dump(glossary_dict, f, ensure_ascii=False, indent=2)

print('Glossary saved to data/glossary.json.')

Glossary entries: 13438
Sample entries: {'a-ba-a-a': 'Abā', 'a-ba-a': 'Abā', 'a-ba': 'Abā', 'ab-ba': 'Abā', 'a-ba-a-a-ma': 'Abāma'}
Glossary saved to data/glossary.json.


In [ ]:
# --- Preprocessing report: what was removed and before/after counts ---
# Counts before preprocessing / deduplication
before = {
    'Competition train': len(train_df),
    'Competition test':  len(test_df),
    'Akkademia train (raw)': len(akkademia_data['train']),
    'Akkademia valid (raw)': len(akkademia_data['valid']),
    'Akkademia test (raw)': len(akkademia_data['test']),
    'ORACC (raw)': len(oracc_df),
    'Sentences_Oare (raw)': len(sentences_oare_df),
}

# Counts after normalization / deduplication / filtering
after = {
    'Akkademia train (deduped)': len(akkademia_deduped['train']),
    'Akkademia valid (deduped)': len(akkademia_deduped['valid']),
    'Akkademia test (deduped)': len(akkademia_deduped['test']),
    'ORACC (deduped)': len(oracc_deduped),
    'Sentences_Oare (filtered)': len(sentences_oare_filtered),
    'Stage 1 train rows': len(stage1_data),
    'Stage 2 train rows': len(stage2_data),
    'Stage 3 train rows': len(stage3_train),
    'Stage 3 val rows': len(stage3_val),
    'Glossary entries': len(glossary_dict),
}

# Rows removed (before - after) with simple reasons
removed = {
    'Akkademia train removed (overlap/dupes)': before['Akkademia train (raw)'] - after['Akkademia train (deduped)'],
    'Akkademia valid removed (overlap/dupes)': before['Akkademia valid (raw)'] - after['Akkademia valid (deduped)'],
    'Akkademia test removed (overlap/dupes)': before['Akkademia test (raw)'] - after['Akkademia test (deduped)'],
    'ORACC removed (overlap/dupes)': before['ORACC (raw)'] - after['ORACC (deduped)'],
    'Sentences_Oare removed (nulls/overlap)': before['Sentences_Oare (raw)'] - after['Sentences_Oare (filtered)'],
    'Stage1 rows removed (missing pairs)': (len(pd.concat([akkademia_stage1[['transliteration_normalized','translation_normalized']], oracc_deduped[['transliteration_normalized','translation_normalized']]])) - len(stage1_data)),
}

import textwrap

print('=' * 80)
print('PREPROCESSING REPORT: BEFORE / AFTER / REMOVED COUNTS')
print('=' * 80)
print('\nBEFORE (loaded datasets):')
print(pd.Series(before).to_string())
print('\nAFTER (post-normalize/dedup/filter):')
print(pd.Series(after).to_string())
print('\nREMOVED (counts and primary causes):')
print(pd.Series(removed).to_string())

# Examples of normalization changes in tabular format
def get_normalization_samples(df, orig_col, norm_col, n=8):
    """Extract samples where normalization changed the text, return as DataFrame."""
    dfc = df.dropna(subset=[orig_col, norm_col]).copy()
    changed = dfc[dfc[orig_col].astype(str) != dfc[norm_col].astype(str)]
    if len(changed) == 0:
        return None
    samples = changed[[orig_col, norm_col]].drop_duplicates().head(n).copy()
    samples.columns = ['Original', 'Normalized']
    samples.index = range(1, len(samples) + 1)
    return samples

print('\n' + '=' * 80)
print('NORMALIZATION EXAMPLES (Original → Normalized)')
print('=' * 80)

# Competition train transliterations
print('\n📜 Competition Train - Transliterations')
comp_translit_samples = get_normalization_samples(train_df, 'transliteration', 'transliteration_normalized', n=8)
if comp_translit_samples is not None:
    display(comp_translit_samples)
else:
    print('  No changes detected.')

# Competition train translations
print('\n📜 Competition Train - Translations')
comp_trans_samples = get_normalization_samples(train_df, 'translation', 'translation_normalized', n=8)
if comp_trans_samples is not None:
    display(comp_trans_samples)
else:
    print('  No changes detected.')

# Akkademia transliterations
print('\n📚 Akkademia Dataset - Transliterations')
akk_translit_samples = get_normalization_samples(akkademia_data['train'], 'transliteration', 'transliteration_normalized', n=8)
if akk_translit_samples is not None:
    display(akk_translit_samples)
else:
    print('  No changes detected.')

# Akkademia translations
print('\n📚 Akkademia Dataset - Translations')
akk_trans_samples = get_normalization_samples(akkademia_data['train'], 'translation', 'translation_normalized', n=8)
if akk_trans_samples is not None:
    display(akk_trans_samples)
else:
    print('  No changes detected.')

# ORACC transliterations
print('\n🏛️ ORACC Dataset - Transliterations')
oracc_translit_samples = get_normalization_samples(oracc_df, 'akkadian', 'transliteration_normalized', n=8)
if oracc_translit_samples is not None:
    display(oracc_translit_samples)
else:
    print('  No changes detected.')

# ORACC translations
print('\n🏛️ ORACC Dataset - Translations')
oracc_trans_samples = get_normalization_samples(oracc_df, 'english', 'translation_normalized', n=8)
if oracc_trans_samples is not None:
    display(oracc_trans_samples)
else:
    print('  No changes detected.')

# Summary of cleaning actions in tabular format
print('\n' + '=' * 80)
print('CLEANING ACTIONS APPLIED (Scribbles & Noise Removed)')
print('=' * 80)

cleaning_df = pd.DataFrame({
    'Category': [
        'Unicode', 'Unicode', 'Damage Markers', 'Scribal Markers',
        'Whitespace', 'Quotation Marks', 'Editorial Marks',
        'Data Quality', 'Data Quality', 'Data Quality'
    ],
    'Action': [
        'Unicode NFC normalization',
        'Subscript digits → ASCII',
        'Bracketed lacunae → <gap>',
        'Remove correction markers',
        'Collapse & trim whitespace',
        'Normalize fancy quotes → ASCII',
        'Strip editorial brackets',
        'Drop rows missing pairs',
        'Remove duplicate transliterations',
        'Filter cross-dataset overlaps'
    ],
    'Example': [
        'ś → ś (composed form)',
        '₁ → 1, ₂ → 2',
        '[broken] → <gap>',
        'text! → text, word? → word',
        '  multiple   spaces  → single space',
        '"curly" → "straight"',
        '[translation] → translation',
        'Rows with null translation removed',
        'Keep first occurrence only',
        'Remove Akkademia/ORACC rows in competition train'
    ]
})

cleaning_df.index = range(1, len(cleaning_df) + 1)
display(cleaning_df)

print('\n✅ All preprocessing complete. Clean datasets saved to data/ directory.')

--- Preprocessing: before / after / removed counts ---
Before (loaded datasets):
Competition train         1561
Competition test             4
Akkademia train (raw)    50478
Akkademia valid (raw)     2812
Akkademia test (raw)      2870
ORACC (raw)               2117
Sentences_Oare (raw)      9782
After (post-normalize/dedup/filter):
Akkademia train (deduped)    44511
Akkademia valid (deduped)     2712
Akkademia test (deduped)      2762
ORACC (deduped)               2111
Sentences_Oare (filtered)     8565
Stage 1 train rows           49334
Stage 2 train rows            8565
Stage 3 train rows            1404
Stage 3 val rows               157
Glossary entries             13438
Removed (counts and primary causes):
Akkademia train removed (overlap/dupes)    5967
Akkademia valid removed (overlap/dupes)     100
Akkademia test removed (overlap/dupes)      108
ORACC removed (overlap/dupes)                 6
Sentences_Oare removed (nulls/overlap)     1217
Stage1 rows removed (missing pairs)   

## 8. Preprocessing Summary

In [ ]:
print('=' * 80)
print('FINAL PREPROCESSING SUMMARY')
print('=' * 80)

# Create comprehensive before/after comparison
summary_data = []

# Competition datasets (not modified)
summary_data.append(['Competition', 'train', len(train_df), len(train_df), 0, 'Source dataset (preserved)'])
summary_data.append(['Competition', 'test', len(test_df), len(test_df), 0, 'Held-out test set'])

# Akkademia datasets
summary_data.append(['Akkademia', 'train', len(akkademia_data['train']), len(akkademia_deduped['train']), 
                     len(akkademia_data['train']) - len(akkademia_deduped['train']), 'Overlap/duplicates removed'])
summary_data.append(['Akkademia', 'valid', len(akkademia_data['valid']), len(akkademia_deduped['valid']), 
                     len(akkademia_data['valid']) - len(akkademia_deduped['valid']), 'Overlap/duplicates removed'])
summary_data.append(['Akkademia', 'test', len(akkademia_data['test']), len(akkademia_deduped['test']), 
                     len(akkademia_data['test']) - len(akkademia_deduped['test']), 'Overlap/duplicates removed'])

# ORACC dataset
summary_data.append(['ORACC', 'train', len(oracc_df), len(oracc_deduped), 
                     len(oracc_df) - len(oracc_deduped), 'Overlap/duplicates removed'])

# Sentences_Oare
summary_data.append(['Sentences_Oare', 'all', len(sentences_oare_df), len(sentences_oare_filtered), 
                     len(sentences_oare_df) - len(sentences_oare_filtered), 'Nulls/overlap removed'])

summary_comparison = pd.DataFrame(summary_data, columns=[
    'Dataset', 'Split', 'Before', 'After', 'Removed', 'Note'
])
summary_comparison.index = range(1, len(summary_comparison) + 1)

print('\nDataset Processing Summary:')
display(summary_comparison)

# Stage dataset summary
print('\n' + '-' * 80)
print('Training Stage Datasets:')
print('-' * 80)

stage_summary = pd.DataFrame({
    'Stage': ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 3'],
    'Purpose': [
        'General Akkadian (external data)',
        'Domain adaptation (OARE sentences)',
        'Fine-tuning (competition train)',
        'Validation split'
    ],
    'Split': ['train', 'train', 'train', 'val'],
    'Rows': [len(stage1_data), len(stage2_data), len(stage3_train), len(stage3_val)],
    'Source': [
        'Akkademia train+valid, ORACC',
        'Sentences_Oare (filtered)',
        'Competition train (90%)',
        'Competition train (10%)'
    ]
})
stage_summary.index = range(1, len(stage_summary) + 1)
display(stage_summary)

print(f'\n📖 Glossary entries (proper names): {len(glossary_dict)}')
print(f'\n💾 All preprocessed datasets saved to data/ directory')
print(f'✅ Preprocessing pipeline complete!')

                           count
Competition train rows      1561
Competition test rows          4
Akkademia train (deduped)  44511
Akkademia valid (deduped)   2712
Akkademia test  (deduped)   2762
ORACC (deduped)             2111
Sentences_Oare (filtered)   8565
Stage 1 train rows         49334
Stage 2 train rows          8565
Stage 3 train rows          1404
Stage 3 val rows             157
Glossary entries           13438
